# **BioGen AI - RFdiffusion Colab Backend**

This notebook acts as the GPU backend for the BioGen AI desktop app.
It installs RFdiffusion, sets up a FastAPI server, and creates a secure Cloudflare tunnel.

### **Instructions:**
1. Go to **Runtime -> Change runtime type** and select **T4 GPU**.
2. Run **Cell 1** to install dependencies.
3. Run **Cell 2** to start the server.
4. Copy the generated **Backend URL** and paste it into your app's Settings!

In [ ]:
#@title 1. Setup RFdiffusion & Dependencies (~3 min)
import os, time, sys

if not os.path.isdir("params"):
  os.system("apt-get install -y aria2")
  os.system("mkdir params")
  os.system("(\
  aria2c -q -x 16 https://files.ipd.uw.edu/krypton/schedules.zip; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; \
  aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; \
  tar -xf alphafold_params_2022-12-06.tar -C params; \
  touch params/done.txt) &")

if not os.path.isdir("RFdiffusion"):
  print("Installing RFdiffusion & dependencies...")
  os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
  os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator fastapi uvicorn python-multipart pydantic nest-asyncio requests")
  os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  os.system("cd RFdiffusion/env/SE3Transformer; pip install .")
  os.system("wget -qnc https://files.ipd.uw.edu/krypton/ananas; chmod +x ananas")

if not os.path.isdir("colabdesign"):
  print("Installing ColabDesign v1.1.1...")
  os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git@v1.1.1")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

if not os.path.isdir("RFdiffusion/models"):
  print("Moving model checkpoints...")
  os.system("mkdir RFdiffusion/models")
  models = ["Base_ckpt.pt","Complex_base_ckpt.pt"]
  for m in models:
    while os.path.isfile(f"{m}.aria2"):
      time.sleep(5)
  os.system(f"mv {' '.join(models)} RFdiffusion/models")
  os.system("unzip -o schedules.zip; rm -f schedules.zip")

if 'RFdiffusion' not in sys.path:
  os.environ["DGLBACKEND"] = "pytorch"
  sys.path.append('RFdiffusion')

import subprocess
print("Patching DGL for Python 3.10+ compatibility...")
result = subprocess.run(["grep", "-rl", "from collections import", "/usr/local/lib/"], capture_output=True, text=True)
for f in result.stdout.strip().split("\n"):
  if f:
    subprocess.run(["sed", "-i", "s/from collections import \(.*\)Mapping\(.*\)/from collections.abc import \1Mapping\2/;s/from collections import \(.*\)Iterable\(.*\)/from collections.abc import \1Iterable\2/", f])

print("\n>>> RFdiffusion Environment Setup Complete! <<<")

In [ ]:
#@title 2. Start BioGen API Server (Cloudflare)
import os, time, uuid, threading, subprocess, secrets, re
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio

# Generate API Key
API_KEY = secrets.token_urlsafe(16)

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class JobRequest(BaseModel):
    name: str
    pdb: str
    num_designs: int
    iterations: int
    contigs: str
    hotspots: str
    symmetry: bool

jobs = {}

def verify_api_key(x_api_key: str = Header(None)):
    if x_api_key and x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API Key")
    return True

@app.get("/api/v1/health")
def health_check():
    return {"status": "ok", "gpu": "connected", "engine": "RFdiffusion + MPNN + AF2"}

@app.post("/api/v1/jobs")
def submit_job(req: JobRequest):
    job_id = str(uuid.uuid4())
    jobs[job_id] = {
        "status": "running",
        "progress_pct": 0,
        "request": req.dict()
    }
    
    def run_full_pipeline(j_id, r):
        try:
            path = f"job_{j_id}"
            full_path = f"outputs/{path}"
            os.makedirs(full_path, exist_ok=True)
            
            # 1. Download PDB if needed
            pdb_id = r["pdb"]
            pdb_file = ""
            if pdb_id and len(pdb_id) == 4:
                os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_id}.pdb1.gz")
                os.system(f"gunzip -f {pdb_id}.pdb1.gz")
                pdb_file = f"{pdb_id}.pdb1"
            
            jobs[j_id]["progress_pct"] = 10
            
            # 2. RFdiffusion (Backbone Generation)
            print(f"\n--- [1/3] RUNNING RFDIFFUSION FOR {j_id} ---")
            cmd1 = f"./RFdiffusion/run_inference.py inference.output_prefix={full_path} inference.num_designs={r['num_designs']} diffuser.T={r['iterations']}"
            if pdb_file:
                cmd1 += f" inference.input_pdb={pdb_file}"
            if r['hotspots']:
                cmd1 += f" 'ppi.hotspot_res=[{r['hotspots']}]'"
                
            contigs = r['contigs']
            cmd1 += f" 'contigmap.contigs=[{contigs}]'"
            
            subprocess.run(cmd1, shell=True, check=True)
            jobs[j_id]["progress_pct"] = 50
            
            # 3. ProteinMPNN + AlphaFold2 (Sequence & Structure Validation)
            print(f"\n--- [2/3] RUNNING PROTEINMPNN & ALPHAFOLD FOR {j_id} ---")
            # Wait for AF2 params to exist
            while not os.path.isfile("params/done.txt"):
                time.sleep(5)
                
            contigs_str = contigs.replace(' ', ':')
            opts = [
                f"--pdb=outputs/{path}_0.pdb",
                f"--loc={full_path}",
                f"--contig={contigs_str}",
                f"--copies=1",
                f"--num_seqs=8",
                f"--num_recycles=1",
                f"--rm_aa=C",
                f"--mpnn_sampling_temp=0.1",
                f"--num_designs={r['num_designs']}"
            ]
            cmd2 = f"python colabdesign/rf/designability_test.py {' '.join(opts)}"
            
            subprocess.run(cmd2, shell=True, check=True)
            
            print(f"\n--- [3/3] PIPELINE COMPLETE FOR {j_id} ---")
            jobs[j_id]["progress_pct"] = 100
            jobs[j_id]["status"] = "completed"
            
        except Exception as e:
            print(f"Error in job {j_id}: {str(e)}")
            jobs[j_id]["status"] = "failed"
            jobs[j_id]["error"] = str(e)

    threading.Thread(target=run_full_pipeline, args=(job_id, req.dict())).start()
    return {"job_id": job_id}

@app.get("/api/v1/jobs/{job_id}")
def get_job_status(job_id: str):
    if job_id not in jobs:
        return {"status": "failed", "error": "Job not found"}
    return jobs[job_id]

def start_cloudflared(port):
    os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    os.system("chmod +x cloudflared-linux-amd64")
    
    process = subprocess.Popen(
        ['./cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    
    url = None
    for line in process.stderr:
        match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break
            
    if url:
        print("\n" + "="*70)
        print(f"✅ API KEY: {API_KEY}")
        print(f"🚀 COPY THIS URL TO YOUR APP SETTINGS:\n\n   {url}/api/v1")
        print("="*70 + "\n")
    else:
        print("Failed to start Cloudflare tunnel.")

# Start Cloudflared tunnel in a separate thread
threading.Thread(target=start_cloudflared, args=(8000,), daemon=True).start()

# Start FastAPI server
nest_asyncio.apply()
uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")
